# 3. Molecular crystal relaxation

**Kernel:** MACE. **Before starting:** complete the preflight in `workshop_demo/README.md`.
Run cells from top to bottom in a fresh kernel. Each setup creates a new results directory.
Timings depend on the allocated hardware; the instructor should measure them before the session.

**Learning goals:** distinguish atomic and cell relaxation; inspect stress and volume; recognize step-limited results.

**Working pattern:** predict a result, run the calculation, inspect the geometry and convergence,
then explain the result to a partner. Energy is reported in eV, length in angstrom, and force in eV/angstrom.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in the repository, workshop folder, or exercise folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
WORKSHOP = next((p for base in _candidates for p in (base, base / 'workshop_demo')
                 if (p / 'workshop_utils.py').is_file()), None)
if WORKSHOP is None:
    raise RuntimeError('Launch Jupyter from the repository or workshop_demo folder.')
if str(WORKSHOP) not in sys.path:
    sys.path.insert(0, str(WORKSHOP))
from workshop_utils import start_exercise, mace_model, relax, smoke_check, signed_angle
DATA, OUTPUT = start_exercise('MolecularCrystals')
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.visualize import view


## 1. Inspect the starting structures
`COWCAS.cif` is the supplied CSD structure; `my_cocrystal.cif` is the supplied generated candidate
for barbituric acid and vanillin. Inspect one at a time. The display helper reconnects molecular
fragments across periodic boundaries for visualization; calculations use the original periodic structure.

Choose COWCAS for the first pass. Comparing total energies of crystals with different compositions
or numbers of atoms does not establish their relative stability.

**Input inspection note:** ASE emitted symmetry/equivalent-site warnings for COWCAS during input checks. Inspect the expanded atom count, cell, and connectivity with the instructor before treating it as a validated reference. The checked reader produced 172 atoms for COWCAS and 64 for the generated co-crystal; matching these counts alone does not establish correctness.


In [ ]:
from crystal_view import repair_fragmented_molecules
INPUT = 'COWCAS.cif'  # Extension: 'my_cocrystal.cif'
initial = read(DATA / INPUT)
print('Atoms:', len(initial), 'Initial volume:', initial.get_volume(), 'A^3')
view(repair_fragmented_molecules(initial), viewer='x3d')


In [ ]:
from mace.calculators import MACECalculator
DEVICE = 'cpu'  # Only select cuda inside a GPU allocation with a compatible environment.
calculator = MACECalculator(model_paths=mace_model('2023-12-10-mace-128-L0_energy_epoch-249.model'),
                            device=DEVICE, default_dtype='float64')


In [ ]:
initial.calc = calculator
smoke_check(initial)


## 2. Compare two optimization choices
Both calculations start from independent copies of the same geometry. Fixed-cell relaxation moves
atoms only; `FrechetCellFilter` exposes cell degrees of freedom to the optimizer as well.
The cell filter couples lattice and atomic relaxation—it is not a second independent calculation of stress.

These bounded runs may stop before convergence. Report that outcome rather than calling every final
structure optimized. The maximum force reported below is atomic; the cell-filter convergence criterion
also includes cell-related components. Inspect stress separately.


In [ ]:
from ase.filters import FrechetCellFilter
from ase.optimize import FIRE
fixed, variable = initial.copy(), initial.copy()
fixed.calc = calculator
variable.calc = calculator
initial_energy = initial.get_potential_energy()
MAX_STEPS = 100
fixed_ok = relax(fixed, FIRE, fmax=0.05, steps=MAX_STEPS, logfile=str(OUTPUT / 'fixed.log'))
variable_ok = relax(FrechetCellFilter(variable), FIRE, fmax=0.05, steps=MAX_STEPS,
                    logfile=str(OUTPUT / 'variable.log'))
for name, atoms, ok in [('fixed', fixed, fixed_ok), ('variable', variable, variable_ok)]:
    print(f'{name}: converged={ok}; energy change={atoms.get_potential_energy()-initial_energy:.4f} eV')
    print(f'Volume change: {100*(atoms.get_volume()/initial.get_volume()-1):.2f}%')
    print('Maximum atomic force:', np.linalg.norm(atoms.get_forces(), axis=1).max(), 'eV/A')
    print('Stress [xx, yy, zz, yz, xz, xy]:', atoms.get_stress(), 'eV/A^3')
    write(OUTPUT / f'{name}.extxyz', atoms)


In [ ]:
view(repair_fragmented_molecules(variable), viewer='x3d')


## Try, explain, and report
1. Did either run hit the step limit? Inspect its log before increasing the limit.
2. Which cell lengths or angles changed? Did molecular connectivity remain plausible?
3. Why can variable-cell relaxation lower energy more than fixed-cell relaxation?
4. Repeat with the other input and compare *changes within each structure*, not raw total energies between them.

**Checkpoint:** record convergence, energy change, volume change, and residual stress.
Model suitability for intermolecular interactions and dispersion must be assessed before interpreting
this as a prediction of crystal stability. An experimental structure can also reflect temperature and pressure
conditions absent from this static relaxation.
